# Figure 3 | Cortical AD-down hdWGCNA modules

Publication-ready analysis notebook for the manuscript figure. Exploratory checks, scratch outputs, and analyses unrelated to the figure panels have been removed. Paths are repository-relative; set `NVU_PROJECT_ROOT` when running from another location.

Analysis scope:
- Cortical AD-down DEG gene set preparation for L2/3 and L4-6 NVUs.
- hdWGCNA module construction, module-trait association, TOM heatmaps, and GO enrichment outputs.

In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath("..", mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure3")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
suppressMessages({
  library(Seurat)
  library(tidyverse)
  library(cowplot)
  library(patchwork)
  library(ggplot2)
  library(plyr)
  library(WGCNA)
  library(hdWGCNA)
  library(harmony)
  library(corrplot)
  library(igraph)
  library(UCell)
  library(future)
  library(future.apply)
  theme_set(theme_cowplot())
})


set.seed(123)
enableWGCNAThreads(nThreads = 24)
plan("multicore", workers = 24)
options(future.globals.maxSize = 200 * 1024^3)

In [ ]:
# ── 1. 读取数据 / branch-clear 结果 ──────────────────────────────
project_dir <- "../results/06_Wgcna"
output <- file.path(project_dir)
dir.create(output, recursive = TRUE, showWarnings = FALSE)

wgcna_name <- "NVU"
branch_clear_rds <- file.path(output, "Cortex_merged-subtype_0330.hdWGCNA_down_L23_L456.branch_clear.rds")
use_branch_clear_result <- file.exists(branch_clear_rds)

genelist <- read.csv(file.path("../results/Figure3/Venn_union_genelist_L23_L456_Down.csv"))
cat("基因列表长度:", length(genelist$gene), "\n")

if (use_branch_clear_result) {
  seurat_obj <- readRDS(branch_clear_rds)
  # 兼容原 notebook 里固定使用的 NVU 名称；实际结果来自 NVU_global_hd。
  if ("NVU_global_hd" %in% names(seurat_obj@misc)) {
    seurat_obj@misc[[wgcna_name]] <- seurat_obj@misc$NVU_global_hd
    seurat_obj@misc$active_wgcna <- wgcna_name
  }
  st_obj1 <- seurat_obj
  cat("已读取 branch-clear hdWGCNA RDS:", branch_clear_rds, "\n")
  cat("所有细胞合并 WGCNA；细胞数:", ncol(seurat_obj), "\n")
} else {
  st_obj <- readRDS(file.path(project_dir, "Cortex_merged-subtype_0330.rds"))
  # 不再按细胞类型/层级拆开；特定 down gene set 在所有细胞合并后做 hdWGCNA。
  st_obj1 <- st_obj
  cat("用于合并 WGCNA 的细胞数:", ncol(st_obj1), "\n")
  if ("celltype_unit" %in% colnames(st_obj1@meta.data)) {
  }
}

In [ ]:
# ── 2. SetupForWGCNA ─────────────────────────────────────────
if (!use_branch_clear_result) {
  seurat_obj <- SetupForWGCNA(
    st_obj1,
    gene_select = "custom",
    features = genelist$gene,
    wgcna_name = wgcna_name
  )
  cat("WGCNA 基因数:", length(seurat_obj@misc[[wgcna_name]]$wgcna_genes), "\n")
} else {
  cat("已加载 branch-clear 结果，跳过 SetupForWGCNA。\n")
}

In [ ]:
if (!use_branch_clear_result) {
  seurat_obj <- SCTransform(
    seurat_obj,
    variable.features.n = NULL,
    variable.features.rv.th = NULL,
    residual.features = genelist$gene,
    verbose = FALSE
  )

  cat("SCT基因数:", nrow(seurat_obj[["SCT"]]), "\n")
  cat("与genelist重叠:", sum(rownames(seurat_obj[["SCT"]]) %in% genelist$gene), "\n")
  seurat_obj <- RunPCA(seurat_obj, assay = "SCT", verbose = FALSE, features = genelist$gene)
  seurat_obj <- FindNeighbors(seurat_obj, reduction = "pca", dims = 1:30)
} else {
  cat("已加载 branch-clear 结果，跳过 SCTransform/PCA。\n")
}

In [ ]:
if (!use_branch_clear_result) {
  metacell_group_by <- intersect(
    c("sample_id", "group", "celltype_unit", "area_m"),
    colnames(seurat_obj@meta.data)
  )
  ident_group <- if ("sample_id" %in% metacell_group_by) "sample_id" else metacell_group_by[1]

  seurat_obj <- MetacellsByGroups(
    seurat_obj = seurat_obj,
    group.by = metacell_group_by,
    k = 25,
    ident.group = ident_group,
    max_shared = 10,
    min_cells = 25
  )
} else {
  cat("已加载 branch-clear 结果，跳过 MetacellsByGroups。\n")
}

In [ ]:
if (!use_branch_clear_result) {
  dat_expr_groups <- sort(unique(as.character(seurat_obj$celltype_unit)))
  dat_expr_groups <- dat_expr_groups[!is.na(dat_expr_groups) & dat_expr_groups != ""]

  seurat_obj <- SetDatExpr(
    seurat_obj,
    group_name = dat_expr_groups,
    group.by = "celltype_unit",
    assay = "SCT",
    use_metacells = TRUE,
    slot = "data"
  )

  seurat_obj <- TestSoftPowers(
    seurat_obj,
    powers = c(4:12),
    networkType = "signed",
    setDatExpr = FALSE
  )

  plot_list <- PlotSoftPowers(seurat_obj, point_size = 5, text_size = 3)
  p <- wrap_plots(plot_list, ncol = 2)
  print(p)

  power_table <- as.data.frame(GetPowerTable(seurat_obj))
  write.csv(power_table, file.path(output, "power_table.csv"), row.names = FALSE)
  saveRDS(seurat_obj, file.path(output, "01_Down_WGcna_cellbin_Cortex_L23-L456_NVU.before_network.rds"))
} else {
  cat("已加载 branch-clear 结果，跳过 TestSoftPowers。最终采用 power=12。\n")
}

In [ ]:
# 先过滤零方差基因
if (!use_branch_clear_result) {
  datExpr <- GetDatExpr(seurat_obj, wgcna_name = wgcna_name)
  cat("当前矩阵维度:", dim(datExpr), "\n")

  gene_mad <- apply(datExpr, 2, mad)
  cat("零MAD基因数:", sum(gene_mad == 0), "\n")
  cat("低MAD基因数 (MAD<0.01):", sum(gene_mad < 0.01), "\n")

  # Branch-clear hdWGCNA: grey保留，PAM关闭，不合并模块。
  seurat_obj <- ConstructNetwork(
    seurat_obj,
    soft_power = 12,
    setDatExpr = FALSE,
    corType = "bicor",
    networkType = "signed",
    TOMType = "signed",
    deepSplit = 4,
    detectCutHeight = 0.9999,
    minModuleSize = 30,
    mergeCutHeight = 0,
    pamStage = FALSE,
    pamRespectsDendro = TRUE,
    tom_outdir = file.path(output, "TOM"),
    tom_name = wgcna_name,
    overwrite_tom = TRUE,
    wgcna_name = wgcna_name
  )

  PlotDendrogram(seurat_obj, main = "L23/L456 down-gene hdWGCNA clustering")
} else {
  cat("已加载 branch-clear 结果，跳过 ConstructNetwork。\n")
}

modules <- GetModules(seurat_obj, wgcna_name = wgcna_name)

In [ ]:
modules <- GetModules(seurat_obj, wgcna_name = wgcna_name)
modules$module <- as.character(modules$module)
if (!"color" %in% colnames(modules)) modules$color <- modules$module
modules$color <- as.character(modules$color)

module_palette <- c(
  grey = "#eaebea",
  turquoise = "#40E0D0",
  blue = "#6e9bc5",
  brown = "#A52A2A",
  yellow = "#ffee6f",
  green = "#199a73",
  red = "#e94829",
  black = "#fd7c01",
  magenta = "#984EA3",
  purple = "#A020F0"
)
modules$plot_color <- ifelse(
  modules$module %in% names(module_palette),
  module_palette[modules$module],
  modules$color
)

if (!is.null(seurat_obj@misc[[wgcna_name]]$branch_clear_gene_tree)) {
  gene_tree <- seurat_obj@misc[[wgcna_name]]$branch_clear_gene_tree
  datExpr <- GetDatExpr(seurat_obj, wgcna_name = wgcna_name)
  gene_tree$labels <- colnames(datExpr)
  plot_colors <- setNames(modules$plot_color, modules$gene_name)[gene_tree$labels]
} else {
  net <- GetNetworkData(seurat_obj, wgcna_name = wgcna_name)
  gene_tree <- net$dendrograms[[1]]
  gene_labels <- gene_tree$labels
  if (is.null(gene_labels)) gene_labels <- modules$gene_name
  plot_colors <- setNames(modules$plot_color, modules$gene_name)[gene_labels]
}

if (any(is.na(plot_colors))) stop("plot_colors 中有 NA，请检查 gene_tree 与 modules 是否匹配。")

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4)
pdf(file.path(output, "Cortex_down_Dendrogram_allgene_branch_clear.pdf"), width = 12, height = 4)
WGCNA::plotDendroAndColors(
  gene_tree,
  as.character(plot_colors),
  groupLabels = "Module colors",
  dendroLabels = FALSE,
  hang = 0.03,
  addGuide = TRUE,
  guideHang = 0.05,
  main = "L23/L456 down-gene hdWGCNA clustering (grey retained, PAM off, no merge)"
)
dev.off()

png(file.path(output, "Cortex_down_Dendrogram_allgene_branch_clear.png"), width = 1800, height = 850, res = 160)
WGCNA::plotDendroAndColors(
  gene_tree,
  as.character(plot_colors),
  groupLabels = "Module colors",
  dendroLabels = FALSE,
  hang = 0.03,
  addGuide = TRUE,
  guideHang = 0.05,
  main = "L23/L456 down-gene hdWGCNA clustering (grey retained, PAM off, no merge)"
)
dev.off()

In [ ]:
Gene2Module <- seurat_obj@misc$NVU$wgcna_modules

In [ ]:
# 修复：branch-clear RDS 里的 module 是 character，hdWGCNA::ModuleEigengenes 需要 factor levels
modules <- GetModules(seurat_obj, wgcna_name = wgcna_name)
module_levels <- names(sort(table(modules$module), decreasing = TRUE))
module_levels <- c(setdiff(module_levels, "grey"), "grey")

modules$module <- factor(as.character(modules$module), levels = module_levels)
modules$color  <- as.character(modules$color)

seurat_obj <- SetModules(seurat_obj, modules, wgcna_name = wgcna_name)

# ME：模块内基因 PCA 的 PC1
seurat_obj <- ScaleData(seurat_obj, assay = DefaultAssay(seurat_obj), verbose = TRUE)

seurat_obj <- ModuleEigengenes(
  seurat_obj,
  assay = DefaultAssay(seurat_obj),
  scale.model.use = "linear",
  wgcna_name = wgcna_name
)

# 合并带编号的 Micro 和 Astro 亚群
seurat_obj$subcelltype <- as.character(seurat_obj$subcelltype)

seurat_obj$subcelltype[seurat_obj$subcelltype %in%
  c("Micro_0", "Micro_1", "Micro_2", "Micro_3", "Micro_4")] <- "Micro"

seurat_obj$subcelltype[seurat_obj$subcelltype %in%
  c("Astro_0", "Astro_1", "Astro_2", "Astro_3", "Astro_4")] <- "Astro"

seurat_obj$subcelltype <- factor(seurat_obj$subcelltype)

# module eigengenes
MEs <- GetMEs(seurat_obj, harmonized = FALSE, wgcna_name = wgcna_name)

# 写入 metadata，方便 DotPlot / 后续统计
MEs_aligned <- MEs[colnames(seurat_obj), , drop = FALSE]
meta_clean <- seurat_obj@meta.data[, !colnames(seurat_obj@meta.data) %in% colnames(MEs_aligned)]
seurat_obj@meta.data <- cbind(meta_clean, MEs_aligned)

seurat_obj@meta.data[1:2, ]

DMEs_all <- FindAllDMEs(
  seurat_obj,
  group.by = "subcelltype",
  wgcna_name = wgcna_name
)

In [ ]:
# get the modules table:
modules <- GetModules(seurat_obj)
mods <- levels(modules$module); mods <- mods[mods != 'grey']

library(tidyr)
plot_df = DMEs_all
plot_df1 = plot_df
plot_df1$group_module = row.names(plot_df1)

logfc_df = plot_df1[,c('avg_log2FC','group','module')]
pvalue_df = plot_df1[,c('p_val_adj','group','module')]

#logfc_df
logfc_wide<-tidyr::spread(logfc_df,group,avg_log2FC) #year为需要分解的变量，gdp为分解后的列的取值
row.names(logfc_wide) = logfc_wide$module
logfc_wide = logfc_wide[,-1]
logfc_wide = data.frame(t(logfc_wide),check.names=F)
# logfc_wide[1:2,]

#pvalue_df
pvalue_wide<-tidyr::spread(pvalue_df,group,p_val_adj) #year为需要分解的变量，gdp为分解后的列的取值
row.names(pvalue_wide) = pvalue_wide$module
pvalue_wide = pvalue_wide[,-1]
pvalue_wide = data.frame(t(pvalue_wide),check.names=F)

# logfc_wide = logfc_wide[rev(c('Con_CA1','Con_Sub','AD_CA1','AD_Sub')),]

# pvalue_wide = pvalue_wide[,colnames(logfc_wide) ]
# pvalue_wide = pvalue_wide[row.names(logfc_wide), ]

#判断显著性
pmat = pvalue_wide## 先复制一个表格


if(!is.null(pmat) ){
    ssmt <- pmat < 0.001
    pmat[ssmt] <- '***'
    smt <- pmat < 0.01 & pmat >= 0.001
    pmat[smt] <- '**'
    tmat <- pmat < 0.05 & pmat >= 0.01
    pmat[tmat] <- '*'
    pmat[!ssmt & !smt & !tmat]<- ''
}else{
    pmat <- F
}

pmat = as.matrix(pmat)
# pmat

library(ComplexHeatmap)


cell_order <- c("EX", "IN", "Astro", "Micro", "Oligo", "Endo", "Pericyte")
cell_order <- cell_order[cell_order %in% rownames(logfc_wide)]

# 先固定行顺序
logfc_wide  <- logfc_wide[cell_order, , drop = FALSE]
pvalue_wide <- pvalue_wide[cell_order, colnames(logfc_wide), drop = FALSE]

# 按每个 module 在哪个 cell type 中 avg_log2FC 最大来排列列
peak_group <- apply(logfc_wide, 2, function(x) {
  rownames(logfc_wide)[which.max(x)]
})

peak_value <- apply(logfc_wide, 2, max, na.rm = TRUE)

module_order <- names(
  sort(
    setNames(match(peak_group, cell_order), colnames(logfc_wide)),
    na.last = TRUE
  )
)

# 同一 cell type 下有多个 module 时，按 logFC 从高到低排
module_order <- module_order[
  order(
    match(peak_group[module_order], cell_order),
    -peak_value[module_order]
  )
]

logfc_wide  <- logfc_wide[, module_order, drop = FALSE]
pvalue_wide <- pvalue_wide[, module_order, drop = FALSE]

pmat <- matrix("", nrow = nrow(pvalue_wide), ncol = ncol(pvalue_wide))
rownames(pmat) <- rownames(pvalue_wide)
colnames(pmat) <- colnames(pvalue_wide)

pmat[pvalue_wide < 0.05]  <- "*"
pmat[pvalue_wide < 0.01]  <- "**"
pmat[pvalue_wide < 0.001] <- "***"

col_fun = circlize::colorRamp2(c(-6, 0,2),  c("#588dd5", "#fbfbfb", "#f5994e") ) #c("#3C5088","white","#A6443D"))
p = Heatmap(logfc_wide ,cluster_rows = F,cluster_columns = F,#row_split = row.type,#column_order = order(colnames(mat_scaled)),
          # left_annotation = left_anno,
            # column_names_gp = gpar(fontsize = 14, angle = 75), # 设置列名字体大小和旋转角度
            cell_fun = function(j, i, x, y, width, height, fill) {
                # 获取pmat对应位置的星号标记
                star_symbol = pmat[i, j]
                grid.text(star_symbol, x, y, gp = gpar(fontsize = 10))
            },
            width = ncol(logfc_wide)*unit(10, "mm"), height = nrow(logfc_wide)*unit(10, "mm"),## 方块cell大小
            col = col_fun,column_names_rot = 75 )

p

pdf('../results/06_Wgcna/02.Cortex_down_group_area.module.cor.pdf',width = 6,height = 6)
p
dev.off()

In [ ]:
library(pheatmap)
tom_matrix <- readRDS(file.path(output, "TOM", "NVU_tom.rds"))
gene_info <- read.csv(file.path(output, "Cortex_down_NVU.Module.csv"))
gene_info <- gene_info[!gene_info$module == "grey", ]

module_levels <- names(sort(table(gene_info$module), decreasing = TRUE))
gene_info$module <- factor(gene_info$module, levels = module_levels)

module_palette <- c(
  turquoise = "#40E0D0",
  blue = "#6e9bc5",
  brown = "#A52A2A",
  yellow = "#FFFF00",
  green = "#00FF00",
  red = "#FF0000",
  black = "#000000",
  magenta = "#FF00FF",
  purple = "#A020F0"
)
module_colors <- list(Module = setNames(module_palette[module_levels], module_levels))
module_colors$Module[is.na(module_colors$Module)] <- module_levels[is.na(module_colors$Module)]

annotation_df <- data.frame(
  Module = gene_info[order(gene_info$module), "module"],
  row.names = gene_info[order(gene_info$module), "gene_name"]
)

gene_info_sorted <- gene_info[order(gene_info$module), ]
gene_order <- gene_info[order(gene_info$module), "gene_name"]
tom_matrix <- tom_matrix[gene_order, gene_order]

clustered_gene_order <- c()
for (module_name in unique(gene_info_sorted$module)) {
  module_genes <- gene_info_sorted$gene_name[gene_info_sorted$module == module_name]
  module_matrix <- tom_matrix[module_genes, module_genes]
  diag(module_matrix) <- NA
  hc <- hclust(as.dist(1 - module_matrix), method = "ward.D2")
  clustered_gene_order <- c(clustered_gene_order, module_genes[hc$order])
}

tom_filtered <- tom_matrix[clustered_gene_order, clustered_gene_order]
diag(tom_filtered) <- NA
tom_filtered <- scale(tom_filtered, center = TRUE, scale = TRUE)

annotation_df_reordered <- annotation_df[clustered_gene_order, , drop = FALSE]
annotation_df_reordered$Module <- factor(annotation_df_reordered$Module, levels = module_levels)

module_reordered <- as.character(annotation_df_reordered$Module)
module_counts_reordered <- rle(module_reordered)$lengths
gaps_positions_reordered <- cumsum(module_counts_reordered)[-length(module_counts_reordered)]

In [ ]:
# Properly defined color palettes (only one should be active)
# mycolors <- c("#3b374c", "#44598e", "#64a0c0", "#7ec4b7", "#deebcd")  # 藏青-浅绿
# mycolors <- c("#073f82", "#1b71b4", "#58a4cf", "#a2cbe3", "#f2f9fe")  # 藏蓝-浅蓝
# mycolors <- c("#eeecdf", "#becdd2", "#6f9ad1", "#44679f", "#3f4f71")  # 藏蓝-水泥灰
# mycolors <- c("#492952", "#82677e", "white", "#59829e", "#1e4668")    # 脏紫-脏蓝
mycolors <- rev(c("#57121d", "#57121d","#57121d", "#57121d", "#57121d", "#57121d", "#d56e5e", "#eaebea", "#5390b5", "#1f294e", "#1f294e", "#1f294e", "#1f294e", "#1f294e", "#1f294e")) # 经典红-蓝
options(repr.plot.width=7, repr.plot.height=6)   ##图像界面宽度显示设置

p <- pheatmap(
  tom_filtered,
  scale = 'row',
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = annotation_df_reordered,
  annotation_col = annotation_df_reordered,
  annotation_colors = module_colors,
  show_rownames = FALSE,
  show_colnames = FALSE,
  gaps_row = gaps_positions_reordered,
  gaps_col = gaps_positions_reordered,
  color = colorRampPalette(mycolors)(100),
  main = "Gene Correlation Matrix (Clustered within Modules)",
  border_color = NA
)
p

In [ ]:
# 定义颜色序列，确保括号闭合
palette_colors <- rev(c(
  "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152", "#8E0152",
  "#C72582",  "#FAEAF2",
   "#EDF5E1","#529624", 
  "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419", "#276419"
))
library(pheatmap)
# 使用 colorRampPalette 创建调色板
color_palette <- colorRampPalette(palette_colors)(100)

# 生成 heatmap
p = pheatmap(
  tom_filtered,
  scale = 'row',
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = annotation_df_reordered,
  annotation_col = annotation_df_reordered,
  annotation_colors = module_colors,
  show_rownames = FALSE,
  show_colnames = FALSE,
  gaps_row = gaps_positions_reordered,
  gaps_col = gaps_positions_reordered,
  # breaks = seq(0, quantile(tom_matrix, 0.1), length.out = 100),
  color = color_palette,
  main = "Gene Correlation Matrix (Clustered within Modules)",
  border_color = NA
)
p

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)
library(ggplot2)

# 1. 准备数据
cat("模块分布:\n")

modules <- unique(gene_info$module)
cat("\n共", length(modules), "个模块\n")

# 2. 基因名转换为ENTREZID
all_genes_entrez <- bitr(
  unique(gene_info$gene_name),
  fromType = "SYMBOL",
  toType   = "ENTREZID",
  OrgDb    = org.Hs.eg.db
)
cat("\n基因转换成功:", nrow(all_genes_entrez), "/", length(unique(gene_info$gene_name)), "\n")

# 合并回gene_info
gene_info_entrez <- gene_info %>%
  dplyr::left_join(all_genes_entrez,
                   by = c("gene_name" = "SYMBOL")) %>%
  dplyr::filter(!is.na(ENTREZID))

# 3. 每个模块做GO-BP富集
go_results <- list()

for(mod in modules) {
  cat("处理模块:", mod, "...")
  
  mod_genes <- gene_info_entrez$ENTREZID[gene_info_entrez$module == mod]
  cat("基因数:", length(mod_genes), "\n")
  
  if(length(mod_genes) < 10) {
    cat("  基因数太少，跳过\n")
    next
  }
  
  tryCatch({
    ego <- enrichGO(
      gene          = mod_genes,
      OrgDb         = org.Hs.eg.db,
      ont           = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff  = 0.05,
      qvalueCutoff  = 0.2,
      readable      = TRUE   # 转回gene symbol
    )
    
    if(nrow(ego@result) > 0) {
      go_results[[mod]] <- ego
      cat("  显著GO terms:", sum(ego@result$p.adjust < 0.05), "\n")
    } else {
      cat("  无显著结果\n")
    }
  }, error = function(e) {
    cat("  错误:", e$message, "\n")
  })
}

cat("\n有结果的模块数:", length(go_results), "\n")

# 4. 合并所有模块结果
go_combined <- dplyr::bind_rows(
  lapply(names(go_results), function(mod) {
    go_results[[mod]]@result %>%
      dplyr::filter(p.adjust < 0.05) %>%
      dplyr::mutate(module = mod) %>%
      dplyr::slice_head(n = 10)   # 每个模块Top10
  })
)

cat("\n合并后总GO terms数:", nrow(go_combined), "\n")

# 5. 可视化1：每个模块Top5 - 点图
top5_per_module <- go_combined %>%
  dplyr::group_by(module) %>%
  dplyr::arrange(p.adjust) %>%
  dplyr::slice_head(n = 5) %>%
  dplyr::ungroup() %>%
  dplyr::mutate(
    Description = stringr::str_wrap(Description, width = 40),
    log_padj    = -log10(p.adjust),
    GeneRatio_num = sapply(GeneRatio, function(x) {
      parts <- strsplit(x, "/")[[1]]
      as.numeric(parts[1]) / as.numeric(parts[2])
    })
  )

# 模块颜色映射
module_colors <- setNames(
  unique(gene_info$color[gene_info$module %in% names(go_results)]),
  unique(gene_info$module[gene_info$module %in% names(go_results)])
)

p_dot <- ggplot(
  top5_per_module,
  aes(x = module, y = reorder(Description, log_padj))
) +
  geom_point(
    aes(size  = Count,
        color = log_padj)
  ) +
  scale_color_gradient(
    low  = "#FEE0D2",
    high = "#E64B35",
    name = "-log10\n(p.adjust)"
  ) +
  scale_size_continuous(
    name  = "Gene Count",
    range = c(2, 8)
  ) +
  facet_wrap(~ module, scales = "free_y", ncol = 3) +
  theme_classic() +
  theme(
    axis.text.x      = element_text(size = 10, face = "bold"),
    axis.text.y      = element_text(size = 8),
    strip.text       = element_text(size = 11, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = NA),
    plot.title       = element_text(hjust = 0.5, face = "bold", size = 13),
    panel.spacing    = unit(1, "lines"),
    legend.position  = "right"
  ) +
  labs(
    title = "GO-BP Enrichment by WGCNA Module (Top 5)",
    x     = "", y = ""
  )
write.csv(go_combined,'../results/06_Wgcna/Cortex_down_NVU.BP.csv')
print(p_dot)
ggsave('../results/06_Wgcna/GO_BP_dotplot_by_module.pdf',
       p_dot,
       width  = 5 * min(length(go_results), 3),
       height = max(length(go_results) / 3 * 4, 10),
       limitsize = FALSE)

## Hub-gene network panels

In [ ]:
library(Matrix)
library(igraph)
library(scales)

module_table <- read.csv('../results/06_Wgcna/Cortex_down_NVU.Module.csv')
modules_df <- module_table %>%
  dplyr::filter(module != "grey") %>%
  dplyr::select(gene_name, module, color) %>%
  dplyr::distinct()

seurat_obj@misc$wgcna_modules <- modules_df

sct_scale_genes <- rownames(seurat_obj@assays$SCT@scale.data)
genes_use <- modules_df$gene_name[modules_df$gene_name %in% rownames(seurat_obj@assays$SCT@data)]
genes_in_sct <- genes_use[genes_use %in% sct_scale_genes]
genes_in_data <- genes_use[genes_use %in% rownames(seurat_obj@assays$SCT@data)]

if (length(genes_in_sct) > 100) {
  expr_for_network <- t(seurat_obj@assays$SCT@scale.data[genes_in_sct, ])
} else {
  expr_for_network <- t(as.matrix(seurat_obj@assays$SCT@data[genes_in_data, ]))
}

cells_common <- intersect(rownames(expr_for_network), rownames(MEs))
expr_net_aligned <- expr_for_network[cells_common, , drop = FALSE]
MEs_aligned <- MEs[cells_common, , drop = FALSE]

mod_colors <- setdiff(unique(modules_df$module), "grey")
kME_list <- list()

for (mod in mod_colors) {
  mod_genes_cur <- modules_df$gene_name[modules_df$module == mod]
  mod_genes_cur <- mod_genes_cur[mod_genes_cur %in% colnames(expr_net_aligned)]
  if (length(mod_genes_cur) == 0 || !mod %in% colnames(MEs_aligned)) next

  expr_mod <- as.matrix(expr_net_aligned[, mod_genes_cur, drop = FALSE])
  me_vec <- MEs_aligned[[mod]]
  kme_vals <- cor(expr_mod, me_vec, use = "pairwise.complete.obs")[, 1]

  kME_list[[mod]] <- data.frame(
    gene_name = names(kme_vals),
    kME = kme_vals,
    module = mod,
    stringsAsFactors = FALSE
  )
}

modules_with_kme <- modules_df %>%
  dplyr::left_join(
    dplyr::bind_rows(kME_list) %>%
      tidyr::pivot_wider(names_from = module, values_from = kME, names_prefix = "kME_"),
    by = "gene_name"
  )

network_out_dir <- file.path(RESULTS_DIR, "06_Wgcna", "module_networks_cortex_down")
dir.create(network_out_dir, recursive = TRUE, showWarnings = FALSE)

for (mod in mod_colors) {
  kme_col <- paste0("kME_", mod)
  if (!kme_col %in% colnames(modules_with_kme)) next

  mod_data <- modules_with_kme %>%
    dplyr::filter(module == mod) %>%
    dplyr::arrange(desc(abs(!!sym(kme_col))))

  top_genes <- head(mod_data$gene_name, min(25, nrow(mod_data)))
  top_genes <- top_genes[top_genes %in% colnames(expr_net_aligned)]
  if (length(top_genes) < 3) next

  expr_top <- as.matrix(expr_net_aligned[, top_genes, drop = FALSE])
  cor_mat <- cor(expr_top, use = "pairwise.complete.obs")
  diag(cor_mat) <- 0

  cor_vals <- abs(cor_mat[upper.tri(cor_mat)])
  thresh <- max(quantile(cor_vals, 0.6, na.rm = TRUE), 0.05)
  cor_mat[abs(cor_mat) < thresh] <- 0

  g <- igraph::graph_from_adjacency_matrix(
    cor_mat, mode = "undirected", weighted = TRUE, diag = FALSE
  )

  kme_vals <- mod_data[[kme_col]][match(igraph::V(g)$name, mod_data$gene_name)]
  kme_vals[is.na(kme_vals)] <- 0.1
  node_size <- scales::rescale(abs(kme_vals), to = c(6, 20))

  n_edges <- igraph::ecount(g)
  edge_width <- if (n_edges > 0 && !is.null(igraph::E(g)$weight)) {
    scales::rescale(abs(igraph::E(g)$weight), to = c(0.5, 5))
  } else {
    rep(1, max(n_edges, 1))
  }

  pdf(file.path(network_out_dir, paste0(mod, "_network.pdf")), width = 9, height = 9)
  par(mar = c(1, 1, 3, 1))
  plot(
    g,
    vertex.size = node_size,
    vertex.color = adjustcolor(mod, alpha.f = 0.85),
    vertex.label = igraph::V(g)$name,
    vertex.label.cex = 0.8,
    vertex.label.color = "black",
    vertex.frame.color = "white",
    edge.width = edge_width,
    edge.color = adjustcolor("grey40", alpha.f = 0.5),
    layout = igraph::layout_with_fr(g),
    main = paste0("Module: ", mod, " (", nrow(mod_data), " genes, ", n_edges, " edges)")
  )
  dev.off()
}